# 03 — Power gate and acquisition audit

Decides **which fairness comparisons these data can support**, and fixes that decision
before any model output exists.

The primary endpoint is a difference in false-negative rate between demographic groups
*within* a view stratum. It is estimated among positive patients only, so precision is
governed by positives per cell — not by cohort size. A site with 60,000 patients can
still be unusable for a label with 44 positives in the cell being compared.

**Runs on CPU in a few minutes; reads only the cohort parquets from notebook 02.**

Outputs
- `outputs/analysis/cell_power_{hash}.csv` — minimum detectable effect per cell
- `outputs/analysis/label_tiers_{hash}.csv` — primary / exploratory / descriptive
- `outputs/analysis/acquisition_coupling_{hash}.csv` — P(AP | age, sex) per site

In [ ]:
# --- setup -------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/pxr-repo'
!git clone -q https://github.com/pridem755/patient-or-xray.git $REPO 2>/dev/null || (cd $REPO && git pull -q)
%pip install -q -e $REPO
# NOTE: if pxr fails to import below, use Runtime -> Restart session, then re-run.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

from pxr.config import load_config
from pxr.stats.power import (
    add_age_group,
    apply_coarsening_ladder,
    assign_tiers,
    cell_positive_counts,
    evaluate_cells,
    minimum_detectable_effect,
)

cfg = load_config(f'{REPO}/config/study_config.yaml')
ROOT = Path(cfg.paths['drive_root'])
COHORTS = ROOT / cfg.paths['cohorts']
OUT = ROOT / cfg.paths['analysis']
OUT.mkdir(parents=True, exist_ok=True)

power_cfg = cfg.analysis['power']
BASELINE = power_cfg['assumed_baseline_fnr']
MEANINGFUL = power_cfg['meaningful_effect']
REPLICATES = power_cfg['simulation_replicates']
LADDER = power_cfg['coarsening_ladder']
SENS_GRID = power_cfg['baseline_sensitivity_grid']
AGE_CUT = cfg.primary_age_threshold

print('config_hash        :', cfg.config_hash)
print('assumed baseline   :', BASELINE, '(FNR level at which the gap is detected)')
print('meaningful gap     :', MEANINGFUL)
print('age contrast       : <', AGE_CUT, ' vs >=', AGE_CUT, ' (fixed clinical cut-point)')
print('inferential sites  :', cfg.inferential_sites)
print('simulation reps    :', f'{REPLICATES:,}')

In [ ]:
cohorts = {}
for site in cfg.site_names:
    path = COHORTS / cfg.artifact_name('cohort', site=site)
    cohorts[site] = pd.read_parquet(path)
    print(f'{site:<10} {len(cohorts[site]):>7,} patients   {path.name}')

assert all(set(df.config_hash) == {cfg.config_hash} for df in cohorts.values()), \
    'cohort config_hash does not match the current config - rebuild in notebook 02'

## 1. What a cell of a given size can detect

Power for a *difference* in rates depends on the level those rates sit at, so the
baseline FNR is a stated assumption rather than an estimate — estimating it would
require the trained model, which does not exist yet. This table is the yardstick for
everything below.

In [ ]:
rng = np.random.default_rng(0)
rows = []
for n in (50, 100, 250, 500, 1000, 2500, 5000):
    mde = minimum_detectable_effect(n, n, baseline=BASELINE, replicates=1500, rng=rng)
    rows.append({'positives per group': f'{n:,}',
                 'smallest detectable gap': f'{mde * 100:.1f} pp'})
print(pd.DataFrame(rows).to_string(index=False))
print(f'\nA {MEANINGFUL:.0%} gap needs roughly 400+ positives in each group.')

## 2. Acquisition coupling — Results §1

The descriptive percentages from notebook 02, formalised: a logistic model of
P(AP | age, sex) per site, reported as odds ratios per decade of age with confidence
intervals. This is the coupling the study is about, and it is estimated on the cohort
itself rather than on model output, so it is available before the prereg freeze.

In [ ]:
coupling = []
for site, df in cohorts.items():
    d = df.assign(is_ap=(df.view == 'AP').astype(int), age_decade=df.age / 10)
    model = smf.logit('is_ap ~ age_decade + C(sex)', data=d).fit(disp=0)
    ci = model.conf_int()
    for term in model.params.index:
        if term == 'Intercept':
            continue
        coupling.append({
            'site': site,
            'term': term,
            'odds_ratio': np.exp(model.params[term]),
            'ci_low': np.exp(ci.loc[term, 0]),
            'ci_high': np.exp(ci.loc[term, 1]),
            'p_value': model.pvalues[term],
        })

coupling = pd.DataFrame(coupling)
print(coupling.round(4).to_string(index=False))
print('\nAn odds ratio above 1 for age_decade means older patients are imaged AP more often.')

In [ ]:
# observed P(AP) by band, alongside the model — the descriptive companion to §1
print('P(AP) by age band:')
print(pd.concat([
    df.assign(site=site).groupby(['site', 'age_bin'], observed=True)
      .view.apply(lambda v: (v == 'AP').mean())
    for site, df in cohorts.items()
]).unstack().round(3).to_string())

## 3. Positives per inferential cell

One row per site x label x view x stratum, with the positive count on each side of the
pre-specified contrast: female vs male, and the fixed clinical age split (under 65 vs
65 and over). The four age bands are reported descriptively elsewhere; they are not the
inferential contrast, because comparing the extreme bands would maximise any monotone
effect while sitting in the thinnest cells.

In [ ]:
counts = pd.concat(
    [cell_positive_counts(df, cfg.analysis_labels, site=site, age_threshold=AGE_CUT)
     for site, df in cohorts.items()],
    ignore_index=True,
)
print(f'{len(counts)} inferential cells (sex and age contrasts x AP/PA x label x site)\n')
print('smallest cells:')
print(counts.assign(smaller=counts[['n_a', 'n_b']].min(axis=1))
            .nsmallest(10, 'smaller')
            .loc[:, ['site', 'label', 'view', 'stratum', 'level_a', 'n_a', 'level_b', 'n_b']]
            .to_string(index=False))

## 4. Evaluate every cell

Each cell's minimum detectable effect is found by simulating the comparison directly
rather than by the normal approximation, which degrades exactly in the small cells
where the decision is hardest. The seed is fixed so the gate returns the same verdict
on every run.

In [ ]:
table = evaluate_cells(
    counts,
    baseline=BASELINE,
    alpha=power_cfg['alpha'],
    target_power=power_cfg['target_power'],
    replicates=REPLICATES,
    meaningful_effect=MEANINGFUL,
    seed=cfg.splits['seed'],
)

print(f'powered cells: {int(table.powered.sum())} / {len(table)}\n')
print('by site:'); print(table.groupby('site').powered.agg(['sum', 'count']).to_string())
print('\nby stratum:'); print(table.groupby('stratum').powered.agg(['sum', 'count']).to_string())

In [ ]:
# per site x label, is the comparison estimable?
pivot = (table.groupby(['label', 'site']).powered.all().unstack()
              .reindex(cfg.analysis_labels))
print('all cells powered (both views, both strata):')
print(pivot.to_string())

## 5. Coarsening ladder

Applied in the pre-specified order. Pooling age bands to a median split roughly doubles
the positives per age cell and is tried first; labels still failing are demoted rather
than dropped, so nothing disappears silently.

In [ ]:
final_table, applied = apply_coarsening_ladder(
    cohorts,
    cfg.analysis_labels,
    LADDER,
    age_threshold=AGE_CUT,
    baseline=BASELINE,
    alpha=power_cfg['alpha'],
    target_power=power_cfg['target_power'],
    replicates=REPLICATES,
    meaningful_effect=MEANINGFUL,
    seed=cfg.splits['seed'],
)

print('rung reached per label:')
for label, rung in applied.items():
    print(f'  {label:<18} {rung}')

## 6. Tier assignment — the gate's verdict

`primary` enters the Holm-corrected family; `exploratory` is Benjamini-Hochberg
corrected and reported as such; `descriptive` is reported without inferential claims.
No Finding is barred from the primary family regardless of power, because its error
semantics invert.

In [ ]:
tiers = assign_tiers(
    final_table,
    cfg.analysis_labels,
    secondary_lane=cfg.secondary_lane,
    inferential_sites=cfg.inferential_sites,
)
print(f'tiers gated on: {cfg.inferential_sites}')
print('(other sites are evaluated and reported, but do not decide the tier)\n')
print(tiers.round(3).to_string(index=False))

primary = tiers.loc[tiers.tier == 'primary', 'label'].tolist()
print(f'\nPRIMARY FAMILY ({len(primary)}): {primary}')
print(f'EXPLORATORY: {tiers.loc[tiers.tier == "exploratory", "label"].tolist()}')
print(f'DESCRIPTIVE: {tiers.loc[tiers.tier == "descriptive", "label"].tolist()}')

## 7. Sensitivity to the assumed baseline

The baseline FNR is an assumption, so the tier assignment must be shown to be robust
to it. If a label changes tier between 20% and 40%, that fragility belongs in the
Methods rather than being hidden by a single choice.

In [ ]:
sensitivity = []
for baseline in SENS_GRID:
    alt = evaluate_cells(counts, baseline=baseline, alpha=power_cfg['alpha'],
                         target_power=power_cfg['target_power'],
                         replicates=max(2000, REPLICATES // 5),
                         meaningful_effect=MEANINGFUL, seed=cfg.splits['seed'])
    alt_tiers = assign_tiers(alt, cfg.analysis_labels,
                             secondary_lane=cfg.secondary_lane,
                             inferential_sites=cfg.inferential_sites)
    sensitivity.append(alt_tiers.set_index('label')['tier'].rename(f'FNR={baseline:.0%}'))

sens = pd.concat(sensitivity, axis=1).reindex(cfg.analysis_labels)
sens['stable'] = sens.nunique(axis=1) == 1
print(sens.to_string())
print('\nA label whose tier moves across plausible baselines is fragile: report that '
      'in Methods rather than letting one assumed value settle it.')

## 8. Save and record

These artifacts are cited by the preregistration in notebook 04. After that freeze, no
analysis choice may be made with results in view.

In [ ]:
final_table.to_csv(OUT / f'cell_power_{cfg.config_hash}.csv', index=False)
tiers.to_csv(OUT / f'label_tiers_{cfg.config_hash}.csv', index=False)
coupling.to_csv(OUT / f'acquisition_coupling_{cfg.config_hash}.csv', index=False)
sens.to_csv(OUT / f'tier_sensitivity_{cfg.config_hash}.csv')
print('saved to', OUT)

In [ ]:
# --- integrity cell: values for the reproducibility appendix ------------------
print(f'config_hash        : {cfg.config_hash}')
print(f'baseline FNR       : {BASELINE}   meaningful gap: {MEANINGFUL}')
print(f'simulation reps    : {REPLICATES}   seed: {cfg.splits["seed"]}')
print(f'cells evaluated    : {len(final_table)}')
print(f'cells powered      : {int(final_table.powered.sum())}')
print(f'primary family     : {primary}')
print(f'tier stable across baselines: {bool(sens.stable.all())}')